# Demo: Runaway AI Query Detection & Cancellation

> **Feature:** Automated detection and cancellation of expensive Cortex AI Function queries  
> **GA Date:** March 2, 2026  
> **Key View:** `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY`  
> **Use Case:** Prevent overnight/long-running AI queries from burning through credit budgets

## Problem Statement

A single AI_COMPLETE query running overnight against a large dataset can accumulate tens of thousands of dollars in credits. Without automated intervention, there's no mechanism to stop a runaway query once it starts consuming tokens.

This demo shows how to implement an automated system that:
1. Detects AI Function queries exceeding a credit threshold while still running
2. Cancels them via `SYSTEM$CANCEL_QUERY()`
3. Sends email alerts with full query details
4. Runs on a configurable schedule (hourly or more frequent)
5. Supports role-based exceptions for approved long-running workloads

## Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                     Snowflake Account                           │
│                                                                 │
│  ┌──────────────┐     ┌──────────────────────────────────────┐  │
│  │ User Query   │────▶│ CORTEX_AI_FUNCTIONS_USAGE_HISTORY    │  │
│  │ (AI_COMPLETE)│     │ • QUERY_ID                           │  │
│  └──────────────┘     │ • CREDITS (accumulating)             │  │
│                       │ • IS_COMPLETED = FALSE               │  │
│                       │ • USER_ID, ROLE_NAMES                │  │
│                       └─────────────────┬────────────────────┘  │
│                                         │                       │
│                                         ▼                       │
│  ┌──────────────────────────────────────────────────────────┐   │
│  │ TASK: MONITOR_RUNAWAY_AI_QUERIES (every 15 min)          │   │
│  │                                                          │   │
│  │  1. Query view for IS_COMPLETED=FALSE + CREDITS > X      │   │
│  │  2. SYSTEM$CANCEL_QUERY() for each offender              │   │
│  │  3. SYSTEM$SEND_EMAIL() with details                     │   │
│  └──────────────────────────────────────────────────────────┘   │
│                           │                                     │
│                           ▼                                     │
│               ┌───────────────────────┐                         │
│               │  Email Alert Sent     │                         │
│               │  to admin@company.com │                         │
│               └───────────────────────┘                         │
└─────────────────────────────────────────────────────────────────┘
```

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Role | `ACCOUNTADMIN` or role with privileges to create tasks, procedures, and notification integrations |
| Warehouse | A warehouse for the monitoring task to run on |
| Email | Verified recipient email addresses for alerts |
| View Access | Access to `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY` |

## Constants

Configure the variables below before running the rest of the notebook.

In [ ]:
DATABASE_NAME = "<YOUR-DATABASE-NAME>"
WAREHOUSE = 'COMPUTE_WH'
CREDIT_THRESHOLD = 50
SCHEDULE_CRON = '*/15 * * * * UTC'
ALERT_RECIPIENTS = "('<EMAIL-ADDRESS>')" # "('<EMAIL-ADDRESS>', '<EMAIL-ADDRESS>')"
ALERT_EMAIL_TO = '<EMAIL-ADDRESS>'
NOTIFICATION_INTEGRATION_NAME = 'ai_cost_alerts'
TASK_NAME = 'MONITOR_RUNAWAY_AI_QUERIES'
PROCEDURE_NAME = 'MONITOR_AND_CANCEL_RUNAWAY_QUERIES'
EXCEPTION_ROLE = 'AI_FUNCTIONS_LONG_RUNNING_ROLE'
LOOKBACK_HOURS = 48

In [ ]:
USE DATABASE {{DATABASE_NAME}};

## Step 1: Create the Notification Integration

This enables the system to send email alerts when queries are cancelled.

In [ ]:
%%sql -r dataframe_1
CREATE OR REPLACE NOTIFICATION INTEGRATION {{NOTIFICATION_INTEGRATION_NAME}}
    TYPE = EMAIL
    ENABLED = TRUE
    ALLOWED_RECIPIENTS = {{ALERT_RECIPIENTS}};

## Step 2: Create the Runaway Query Detection & Cancellation Procedure

This stored procedure:
- Scans `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` for queries still running (`IS_COMPLETED = FALSE`)
- Filters to queries exceeding a configurable credit threshold
- Cancels each offending query
- Sends a detailed email alert per cancelled query

In [ ]:
USE SCHEMA PUBLIC;

CREATE OR REPLACE PROCEDURE {{PROCEDURE_NAME}}(
    P_CREDIT_THRESHOLD NUMBER DEFAULT {{CREDIT_THRESHOLD}}
)
RETURNS TABLE (
    QUERY_ID VARCHAR,
    USER_NAME VARCHAR,
    FUNCTION_NAME VARCHAR,
    MODEL_NAME VARCHAR,
    CREDITS NUMBER,
    START_TIME TIMESTAMP_LTZ,
    ROLE_NAMES ARRAY,
    QUERY_TAG VARCHAR,
    WAREHOUSE_ID NUMBER,
    ACTION VARCHAR
)
LANGUAGE SQL
AS
$$
DECLARE
    result RESULTSET;
BEGIN
    result := (
        SELECT
            h.QUERY_ID,
            u.NAME AS USER_NAME,
            h.FUNCTION_NAME,
            h.MODEL_NAME,
            h.CREDITS,
            h.START_TIME,
            h.ROLE_NAMES,
            h.QUERY_TAG,
            h.WAREHOUSE_ID,
            'CANCELLED' AS ACTION
        FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
        LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u
            ON h.USER_ID = u.USER_ID
        WHERE h.START_TIME >= DATEADD('hour', -{{LOOKBACK_HOURS}}, CURRENT_TIMESTAMP())
        AND h.CREDITS > :P_CREDIT_THRESHOLD
        AND h.IS_COMPLETED = FALSE
    );

    FOR rec IN result DO
        BEGIN
            EXECUTE IMMEDIATE 'SELECT SYSTEM$CANCEL_QUERY(''' || rec.QUERY_ID || ''')';
        EXCEPTION
            WHEN OTHER THEN
                NULL;
        END;

        CALL SYSTEM$SEND_EMAIL(
            '{{NOTIFICATION_INTEGRATION_NAME}}',
            '{{ALERT_EMAIL_TO}}',
            'Runaway AI Query Cancelled - ' || rec.QUERY_ID,
            'A runaway AI Function query has been cancelled due to excessive cost.\n\n' ||
            'Query Details:\n' ||
            '- Query ID: ' || rec.QUERY_ID || '\n' ||
            '- User: ' || COALESCE(rec.USER_NAME, 'Unknown') || '\n' ||
            '- Function: ' || rec.FUNCTION_NAME || '\n' ||
            '- Model: ' || rec.MODEL_NAME || '\n' ||
            '- Credits Used: ' || rec.CREDITS::VARCHAR || '\n' ||
            '- Threshold: ' || :P_CREDIT_THRESHOLD::VARCHAR || '\n' ||
            '- Start Time: ' || rec.START_TIME::VARCHAR || '\n' ||
            '- Roles: ' || COALESCE(rec.ROLE_NAMES::VARCHAR, 'N/A') || '\n' ||
            '- Query Tag: ' || COALESCE(rec.QUERY_TAG, 'N/A') || '\n' ||
            '- Warehouse ID: ' || COALESCE(rec.WAREHOUSE_ID::VARCHAR, 'N/A') || '\n\n' ||
            'Please investigate this query and take appropriate action.'
        );
    END FOR;

    RETURN TABLE(result);
END;
$$;

## Step 3: Schedule the Monitoring Task

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE TASK {{TASK_NAME}}
    WAREHOUSE = {{WAREHOUSE}}
    SCHEDULE = 'USING CRON {{SCHEDULE_CRON}}'
AS
    CALL {{PROCEDURE_NAME}}({{CREDIT_THRESHOLD}});

ALTER TASK {{TASK_NAME}} RESUME;

## Step 4: Verify the Task is Running

In [ ]:
%%sql -r dataframe_4
SHOW TASKS LIKE '{{TASK_NAME}}';

In [ ]:
%%sql -r dataframe_5
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -1, CURRENT_TIMESTAMP()),
    TASK_NAME => '{{TASK_NAME}}'
))
ORDER BY SCHEDULED_TIME DESC;

## Step 5: (Optional) Role-Based Exceptions for Approved Long-Running Workloads

Some teams may have legitimate long-running AI workloads (e.g., batch document processing). Create an exception role so these queries aren't auto-cancelled.

In [ ]:
CREATE ROLE IF NOT EXISTS {{EXCEPTION_ROLE}};

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE {{EXCEPTION_ROLE}};

-- Assign to users who need long-running AI queries
-- GRANT ROLE {{EXCEPTION_ROLE}} TO USER BATCH_PROCESSING_USER;

To exclude the exception role, add this condition to the procedure's `WHERE` clause:

```sql
AND NOT ARRAY_CONTAINS(
    '<EXCEPTION_ROLE>'::VARIANT,
    h.ROLE_NAMES
)
```

Users invoke the exception role before running approved workloads:

```sql
USE ROLE <EXCEPTION_ROLE>;
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE('claude-3.5-sonnet', prompt)
FROM large_document_table;
```

## Step 6: Test the Setup

This will attempt to cancel any currently-running AI query and send an alert email. Verify you receive the email, then restore the production threshold.

### Simulate a runaway query (safe test example)

Note: This needs to be modified so it runs for 15-30 minutes then can be caught by the procedure.

```sql
-- Run in a separate session to generate a long-running Cortex AI query
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
    'claude-sonnet-4-6',
    'Write a detailed 500-word essay about the history of artificial intelligence, covering key milestones from the 1950s to present day.'
)
FROM TABLE(GENERATOR(ROWCOUNT => 50));
```



### Test with a threshold of 0 (triggers immediately)

Then we call the procedure to catch any querying in progress (aka - threshold of zero)

In [ ]:
%%sql -r dataframe_6
CALL {{PROCEDURE_NAME}}(0);

## Monitoring Dashboard Queries

### View all cancelled queries (historical)

In [ ]:
%%sql -r dataframe_7
SELECT
    h.QUERY_ID,
    u.NAME AS USER_NAME,
    h.FUNCTION_NAME,
    h.MODEL_NAME,
    h.CREDITS,
    h.START_TIME,
    h.END_TIME,
    h.QUERY_TAG,
    h.ROLE_NAMES
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u
    ON h.USER_ID = u.USER_ID
WHERE h.START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
AND h.CREDITS > {{CREDIT_THRESHOLD}}
ORDER BY h.CREDITS DESC;

### View currently running AI queries and their credit accumulation

In [ ]:
%%sql -r dataframe_8
SELECT
    h.QUERY_ID,
    u.NAME AS USER_NAME,
    h.FUNCTION_NAME,
    h.MODEL_NAME,
    h.CREDITS,
    h.START_TIME,
    DATEDIFF('minute', h.START_TIME, CURRENT_TIMESTAMP()) AS RUNNING_MINUTES,
    h.QUERY_TAG
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY h
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u
    ON h.USER_ID = u.USER_ID
WHERE h.IS_COMPLETED = FALSE
AND h.START_TIME >= DATEADD('hour', -{{LOOKBACK_HOURS}}, CURRENT_TIMESTAMP())
ORDER BY h.CREDITS DESC;

### Daily credit consumption trend (spot anomalies)

In [ ]:
%%sql -r dataframe_9
SELECT
    DATE_TRUNC('day', START_TIME) AS usage_date,
    FUNCTION_NAME,
    MODEL_NAME,
    SUM(CREDITS) AS total_credits,
    COUNT(DISTINCT QUERY_ID) AS query_count,
    MAX(CREDITS) AS max_single_query_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY 1, 2, 3
ORDER BY usage_date DESC, total_credits DESC;

## Important Considerations

| Topic | Detail |
|-------|--------|
| **Latency** | `CORTEX_AI_FUNCTIONS_USAGE_HISTORY` has up to 60 minutes of latency (often available in ~10 min). A query could accumulate credits during this window before detection. |
| **Billing** | Cancelling a query does **not** refund credits already consumed. You're charged up to the moment of cancellation. |
| **Threshold tuning** | Start conservative (e.g., 50 credits ≈ ~$150-200 depending on model). Adjust based on your legitimate workload patterns. |
| **Task frequency** | Every 15 minutes is aggressive but limits max overshoot. Factor in the usage view latency — a 15-min task + 10-min view latency = ~25 min worst case detection time. |
| **Warehouse cost** | The monitoring task itself uses a warehouse. An XS warehouse running for seconds every 15 min is negligible cost. |

## Cleanup / Teardown

In [ ]:
%%sql -r dataframe_11
ALTER TASK {{TASK_NAME}} SUSPEND;

DROP TASK IF EXISTS {{TASK_NAME}};
DROP PROCEDURE IF EXISTS {{PROCEDURE_NAME}}(NUMBER);
DROP NOTIFICATION INTEGRATION IF EXISTS {{NOTIFICATION_INTEGRATION_NAME}};
DROP ROLE IF EXISTS {{EXCEPTION_ROLE}};

## Reference

- [Managing Cortex AI Function costs with Account Usage](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)
- [CORTEX_AI_FUNCTIONS_USAGE_HISTORY view](https://docs.snowflake.com/en/sql-reference/account-usage/cortex_ai_functions_usage_history)
- [SYSTEM$CANCEL_QUERY](https://docs.snowflake.com/en/sql-reference/functions/system_cancel_query)
- [Creating Tasks](https://docs.snowflake.com/en/sql-reference/sql/create-task)